# 05_02 Feature ablation for daily LightGBM Tweedie

Measure the incremental value of feature groups for the global daily Tweedie model. The **base specification** contains only IDs, calendar and forecast-position features, and local demand history. Each candidate specification adds exactly one remaining feature group to that base; the final specification restores all available features. This isolates each group's contribution without allowing later groups to mask earlier ones.

## Experimental controls

All specifications use the same assessed series, 20 weekly test origins, seven-day horizon, an expanding fitting window that starts with 48 origins, four-origin validation window, 28-day refit cadence, Tweedie variance power (1.5), hyperparameters, and WAPE early stopping as `05_01`. New feature snapshots and forecasts are produced every seven days. Consequently, the only intentional difference between rows in the ablation table is the included feature set.

`Base + group` rows estimate isolated incremental value relative to the base. The all-features row measures the joint result, which need not equal the sum of isolated effects because LightGBM can learn interactions and substitute correlated signals.

In [ ]:
from pathlib import Path
import os
import sys

os.environ.setdefault('MPLCONFIGDIR', '/tmp/ba-matplotlib')

import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'src').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.models.benchmark import load_benchmark_design, segment_wape, summarize_models
from src.models.lightgbm import FEATURE_COLUMNS
from src.models.lightgbm.ablation.model import (
    BASE_GROUPS, CANDIDATE_GROUPS, build_feature_sets,
)
from src.models.results import load_ablation_results

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 160)
sns.set_theme(style='whitegrid', context='notebook')


## Feature specifications

The requested three groups form the base. Known actions, historical action behavior, maturity, annual history, demand regime, and cross-sectional context are tested separately. Any later-added production features not explicitly listed in the runner are retained automatically as one `Operational supply signals` group, so the notebook remains aligned with `FEATURE_COLUMNS`.

In [ ]:
CANDIDATE_GROUPS = CANDIDATE_GROUPS.copy()
declared_features = {
    feature
    for group in [*BASE_GROUPS.values(), *CANDIDATE_GROUPS.values()]
    for feature in group
}
operational_features = tuple(
    feature for feature in FEATURE_COLUMNS if feature not in declared_features
)
if operational_features:
    CANDIDATE_GROUPS['Operational supply signals'] = operational_features

FEATURE_SETS = build_feature_sets()

feature_design = pd.DataFrame([
    {
        'specification': specification,
        'n_features': len(features),
        'added_to_base': (
            'None' if specification == 'Base'
            else 'All remaining groups' if specification == 'All features'
            else specification.removeprefix('Base + ')
        ),
        'features': ', '.join(features),
    }
    for specification, features in FEATURE_SETS.items()
])
display(feature_design)

## Shared data and expanding-origin frames

In [ ]:
design = load_benchmark_design()
persisted_feature_design, model_results = load_ablation_results(design)
ablation_forecasts = pd.concat(
    [result.forecasts for result in model_results.values()], ignore_index=True,
)
training_summary = pd.concat(
    [result.training_summary for result in model_results.values()], ignore_index=True,
)
feature_importance = pd.concat(
    [result.feature_importance for result in model_results.values()], ignore_index=True,
)
MODEL_LABELS = dict(zip(
    persisted_feature_design.model, persisted_feature_design.specification,
))

display(pd.DataFrame([{
    'evaluation_origins': ablation_forecasts.origin.nunique(),
    'evaluation_rows_per_model': ablation_forecasts.groupby('model').size().iloc[0],
    'ablation_specifications': len(model_results),
    'model_refits': training_summary.evaluation_origin.nunique() * len(model_results),
}]))


## Fit every feature specification

Categorical encoding is restricted to categorical columns present in each feature set. Early stopping chooses the number of boosting rounds on the validation origins, after which the shared trainer refits on training plus validation rows.

In [ ]:
display(training_summary.groupby('specification', sort=False).agg(
    refits=('evaluation_origin', 'nunique'),
    features=('features', 'first'),
    median_best_iteration=('best_iteration', 'median'),
    min_best_iteration=('best_iteration', 'min'),
    max_best_iteration=('best_iteration', 'max'),
).reset_index().style.format({
    'features': '{:,.0f}', 'refits': '{:,.0f}',
    'median_best_iteration': '{:,.0f}', 'min_best_iteration': '{:,.0f}',
    'max_best_iteration': '{:,.0f}',
}))


## Overall ablation result

Positive `WAPE gain vs base` means that adding the group improves accuracy. Calibration is shown alongside accuracy to distinguish a genuine error reduction from systematic suppression or inflation of forecast volume.

In [ ]:
summary = summarize_models(ablation_forecasts)
summary['specification'] = summary.model.map(MODEL_LABELS)
base_row = summary.loc[summary.specification.eq('Base')].iloc[0]
summary['wape_gain_vs_base_pp'] = 100 * (base_row.pooled_wape - summary.pooled_wape)
summary['absolute_error_reduction_vs_base_kg'] = (
    base_row.absolute_error_kg - summary.absolute_error_kg
    if 'absolute_error_kg' in summary
    else summary.actual_kg * (base_row.pooled_wape - summary.pooled_wape)
)
summary['feature_count'] = summary.specification.map(
    {name: len(features) for name, features in FEATURE_SETS.items()}
)
summary = summary.sort_values('pooled_wape').reset_index(drop=True)
display(summary[[
    'specification', 'feature_count', 'pooled_wape', 'wape_gain_vs_base_pp',
    'relative_bias', 'forecast_to_actual_ratio', 'median_series_wape',
    'seasonal_mase', 'mae_kg', 'actual_kg', 'forecast_kg',
]].style.format({
    'feature_count': '{:,.0f}', 'pooled_wape': '{:.2%}',
    'wape_gain_vs_base_pp': '{:+.2f}', 'relative_bias': '{:+.2%}',
    'forecast_to_actual_ratio': '{:.3f}', 'median_series_wape': '{:.2%}',
    'seasonal_mase': '{:.3f}', 'mae_kg': '{:.3f}',
    'actual_kg': '{:,.1f}', 'forecast_kg': '{:,.1f}',
}).background_gradient(subset=['wape_gain_vs_base_pp'], cmap='RdYlGn'))

In [ ]:
wape_plot = summary.sort_values('pooled_wape', ascending=False).copy()
wape_plot['is_base'] = wape_plot.specification.eq('Base')
wape_plot['is_all_features'] = wape_plot.specification.eq('All features')
wape_colors = np.select(
    [wape_plot.is_base, wape_plot.is_all_features],
    ['#d97706', '#0f766e'],
    default='#64748b',
)
fig, ax = plt.subplots(figsize=(11, 7), constrained_layout=True)
bars = ax.barh(
    wape_plot.specification, wape_plot.pooled_wape, color=wape_colors,
)
ax.axvline(base_row.pooled_wape, color='#d97706', linestyle='--', linewidth=1.5)
ax.xaxis.set_major_formatter(PercentFormatter(1))
ax.set_xlabel('Pooled WAPE')
ax.set_ylabel('')
ax.set_title('Tweedie accuracy by feature specification', loc='left', weight='bold')
ax.bar_label(
    bars, labels=[f'{value:.2%}' for value in wape_plot.pooled_wape],
    padding=4, fontsize=9,
)
ax.margins(x=.12)
sns.despine(fig=fig)
plt.show()

In [ ]:
plot_summary = summary.sort_values('wape_gain_vs_base_pp')
colors = np.where(plot_summary.wape_gain_vs_base_pp.ge(0), '#15803d', '#b91c1c')
fig, axes = plt.subplots(1, 2, figsize=(15, 8), constrained_layout=True)

axes[0].barh(
    plot_summary.specification, plot_summary.wape_gain_vs_base_pp, color=colors,
)
axes[0].axvline(0, color='#222222', linewidth=1)
axes[0].set_xlabel('Pooled WAPE gain vs base (percentage points)')
axes[0].set_ylabel('')
axes[0].set_title('Incremental accuracy', loc='left', weight='bold')
for y, value in enumerate(plot_summary.wape_gain_vs_base_pp):
    axes[0].text(
        value + (0.03 if value >= 0 else -0.03), y, f'{value:+.2f}',
        ha='left' if value >= 0 else 'right', va='center', fontsize=9,
    )

axes[1].axvline(1, color='#222222', linewidth=1)
axes[1].scatter(
    plot_summary.forecast_to_actual_ratio, plot_summary.specification,
    c=colors, s=75,
)
axes[1].set_xlabel('Forecast-to-actual ratio')
axes[1].set_ylabel('')
axes[1].set_title('Volume calibration', loc='left', weight='bold')
for y, value in enumerate(plot_summary.forecast_to_actual_ratio):
    axes[1].annotate(
        f'{value:.3f}', (value, y), xytext=(6, 0),
        textcoords='offset points', va='center', fontsize=9,
    )
sns.despine(fig=fig)
plt.show()

## Stability across evaluation origins

A feature group is more credible when its aggregate gain is repeated across origins rather than driven by one unusually favorable window.

In [ ]:
origin_result = segment_wape(ablation_forecasts, 'origin')
origin_result['specification'] = origin_result.model.map(MODEL_LABELS)
origin_wide = origin_result.pivot(
    index='origin', columns='specification', values='pooled_wape',
)
origin_delta = origin_wide.sub(origin_wide['Base'], axis=0).drop(columns='Base')
origin_stability = pd.DataFrame({
    'specification': origin_delta.columns,
    'origins_improved': origin_delta.lt(0).sum().to_numpy(),
    'origins_evaluated': origin_delta.notna().sum().to_numpy(),
    'mean_wape_gain_pp': (-100 * origin_delta.mean()).to_numpy(),
    'median_wape_gain_pp': (-100 * origin_delta.median()).to_numpy(),
    'worst_wape_gain_pp': (-100 * origin_delta.max()).to_numpy(),
}).sort_values('mean_wape_gain_pp', ascending=False)
display(origin_stability.style.format({
    'origins_improved': '{:,.0f}', 'origins_evaluated': '{:,.0f}',
    'mean_wape_gain_pp': '{:+.2f}', 'median_wape_gain_pp': '{:+.2f}',
    'worst_wape_gain_pp': '{:+.2f}',
}))

fig, ax = plt.subplots(figsize=(15, 7), constrained_layout=True)
sns.heatmap(
    -100 * origin_delta.T, center=0, cmap='RdYlGn', annot=True, fmt='.2f',
    linewidths=.5, cbar_kws={'label': 'WAPE gain vs base (pp)'}, ax=ax,
)
ax.set_xlabel('Evaluation origin')
ax.set_ylabel('')
ax.set_title('Origin-level feature-group gains', loc='left', weight='bold')
plt.show()

origin_gain_long = (
    (-100 * origin_delta)
    .rename_axis(index='origin', columns='specification')
    .stack().rename('wape_gain_pp').reset_index()
)
origin_order = (
    origin_gain_long.groupby('specification', observed=True).wape_gain_pp
    .median().sort_values(ascending=False).index
)
fig, ax = plt.subplots(figsize=(14, 7), constrained_layout=True)
sns.boxplot(
    data=origin_gain_long, x='wape_gain_pp', y='specification',
    order=origin_order, color='#a7f3d0', fliersize=0, ax=ax,
)
sns.stripplot(
    data=origin_gain_long, x='wape_gain_pp', y='specification',
    order=origin_order, color='#134e4a', alpha=.65, size=5, ax=ax,
)
ax.axvline(0, color='#222222', linewidth=1)
ax.set_xlabel('Origin-level WAPE gain vs base (percentage points)')
ax.set_ylabel('')
ax.set_title('Distribution of gains across origins', loc='left', weight='bold')
sns.despine(fig=fig)
plt.show()

## Which added features are used?

Gain importance is averaged over the five four-week refits. Importance does not establish causal value, but it helps explain whether an added group contributes through one dominant signal or several complementary signals.

In [ ]:
added_feature_rows = []
for specification, group_features in CANDIDATE_GROUPS.items():
    label = f'Base + {specification}'
    panel = feature_importance.loc[
        feature_importance.specification.eq(label)
        & feature_importance.feature.isin(group_features)
    ]
    grouped = panel.groupby('feature', observed=True).agg(
        mean_gain_share=('gain_share', 'mean'),
        mean_splits=('splits', 'mean'),
    ).reset_index()
    grouped['added_group'] = specification
    added_feature_rows.append(grouped)

added_feature_importance = pd.concat(added_feature_rows, ignore_index=True)
added_feature_importance = added_feature_importance.sort_values(
    ['added_group', 'mean_gain_share'], ascending=[True, False],
)
display(added_feature_importance.style.format({
    'mean_gain_share': '{:.2%}', 'mean_splits': '{:,.1f}',
}))

importance_plot = (
    added_feature_importance.sort_values(
        ['added_group', 'mean_gain_share'], ascending=[True, False],
    )
    .groupby('added_group', observed=True, sort=False).head(5)
)
groups = list(importance_plot.added_group.drop_duplicates())
ncols = 2
nrows = int(np.ceil(len(groups) / ncols))
fig, axes = plt.subplots(
    nrows, ncols, figsize=(15, 4.2 * nrows), squeeze=False,
    constrained_layout=True,
)
for ax, group in zip(axes.flat, groups):
    panel = (
        importance_plot.loc[importance_plot.added_group.eq(group)]
        .sort_values('mean_gain_share')
    )
    bars = ax.barh(panel.feature, panel.mean_gain_share, color='#0f766e')
    ax.xaxis.set_major_formatter(PercentFormatter(1))
    ax.set_xlabel('Mean gain share')
    ax.set_ylabel('')
    ax.set_title(group, loc='left', weight='bold')
    ax.bar_label(
        bars, labels=[f'{value:.1%}' for value in panel.mean_gain_share],
        padding=3, fontsize=9,
    )
    ax.margins(x=.18)
for ax in axes.flat[len(groups):]:
    ax.set_visible(False)
fig.suptitle(
    'Top added features within each ablation group', fontsize=15, weight='bold',
)
sns.despine(fig=fig)
plt.show()

## Selection rule

Prefer feature groups that improve pooled WAPE, improve a majority of origins, and do not introduce unacceptable volume bias. Treat the all-features result as a separate interaction check rather than evidence that every included group is useful. Any final feature selection should be fixed here before it is evaluated against downstream architectures or a new holdout period.